# Google Merchandise Store — Product Analytics

BigQuery SQL + Python portfolio analysis.

**Workflow:** data quality → strict sequential funnel → segmentation → retention → statistics → experiment design.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower


## 1. Dataset and data quality

In [ ]:
profile = pd.read_csv("../data/profile.csv")
quality = pd.read_csv("../data/data_quality.csv")
display(profile)
display(quality)


The composite `(fullVisitorId, visitId, visitStartTime)` is unique for all 903,653 session rows.  
The 898 repeated `(fullVisitorId, visitId)` pairs have different dates/start times and are therefore retained as distinct sessions.


## 2. Strict sequential funnel

In [ ]:
funnel = pd.read_csv("../data/funnel_sequential.csv")
funnel


![Strict funnel](../images/funnel_sequential.png)

The largest internal drop-off is **Product view → Add to cart: 64.53%**.

Two purchase metrics are kept separate:
- observed purchase-event CR = **1.28%** (11,552 / 903,653);
- strict full-path completion = **1.00%** (9,072 / 903,653).


## 3. Device and user-type segmentation

In [ ]:
device = pd.read_csv("../data/device_funnel_sequential.csv")
user_type = pd.read_csv("../data/user_type_device.csv")
display(device)
display(user_type)


![Observed purchase conversion](../images/device_observed_purchase_conversion.png)

![New vs returning](../images/user_type_device_conversion.png)

Observed purchase conversion is 1.58% on desktop vs 0.41% on mobile.  
The device gap persists inside both new and returning users, so it is not explained only by user-type mix.


## 4. Acquisition channels

In [ ]:
channels = pd.read_csv("../data/channel_performance.csv")
channels


![Channels](../images/channel_efficiency.png)

Referral produces ~46% of purchases from 11.6% of sessions and has 5.08% purchase CR.  
Social contributes 25.0% of sessions but only 0.05% purchase CR.

This is not an ROI comparison: cost and revenue context are incomplete.


## 5. Cohort retention

In [ ]:
retention = pd.read_csv("../data/retention.csv", parse_dates=["cohort_month"])
retention.head()


![Cohort heatmap](../images/cohort_retention_heatmap.png)

![Retention curve](../images/retention_curve.png)

Weighted retention: **M1 3.39% → M2 1.09% → M3 0.58%**.

## 6. Statistical comparison

In [ ]:
desktop_purchases, desktop_sessions = 10528, 664479
mobile_purchases, mobile_sessions = 856, 208725

count = np.array([desktop_purchases, mobile_purchases])
nobs = np.array([desktop_sessions, mobile_sessions])

z_stat, p_value = proportions_ztest(count, nobs)
desktop_cr = desktop_purchases / desktop_sessions
mobile_cr = mobile_purchases / mobile_sessions
diff = desktop_cr - mobile_cr

se = math.sqrt(
    desktop_cr * (1 - desktop_cr) / desktop_sessions +
    mobile_cr * (1 - mobile_cr) / mobile_sessions
)

pd.DataFrame([{
    "desktop_cr_pct": desktop_cr * 100,
    "mobile_cr_pct": mobile_cr * 100,
    "absolute_difference_pp": diff * 100,
    "conversion_ratio": desktop_cr / mobile_cr,
    "z_statistic": z_stat,
    "p_value": p_value,
    "ci95_low_pp": (diff - 1.96*se) * 100,
    "ci95_high_pp": (diff + 1.96*se) * 100,
}]).round(4)


This is an observational comparison, **not an A/B test**. Statistical significance does not establish causality.


## 7. Experiment design

In [ ]:
eligible_users = 23102
converted_users = 6068
baseline = converted_users / eligible_users
target = baseline + 0.05

effect_size = abs(proportion_effectsize(baseline, target))
n_per_group = math.ceil(
    NormalIndPower().solve_power(
        effect_size=effect_size,
        alpha=0.05,
        power=0.80,
        ratio=1,
        alternative="two-sided"
    )
)
buffered = math.ceil(n_per_group * 1.10)

pd.DataFrame([{
    "eligible_users": eligible_users,
    "baseline_pct": baseline * 100,
    "target_pct": target * 100,
    "n_per_group": n_per_group,
    "n_per_group_with_buffer": buffered,
    "total_with_buffer": 2 * buffered
}])


User-level baseline = **26.27%**.  
For an illustrative **+5 p.p. MDE**, the design needs **1,285 users/group**, or **1,414/group** with a 10% buffer.

See [`docs/experiment_design.md`](../docs/experiment_design.md).


## 8. Limitations

- Public, historical, anonymized dataset.
- Observational segment differences are not causal.
- Earliest retention cohorts have left-censoring risk.
- Cohorts have unequal follow-up.
- Channel CR is not marketing ROI.
- Experiment section is design only; no randomized treatment exists in the data.
